# 🫁 ThoraxLens · Clasificación de tomografías de tórax (cáncer de pulmón)
### UDD · IA para la Gestión Industrial

**Problema.** Clasificar un corte de tomografía computarizada (TC) de tórax en **3 clases**: *normal*, *benigno* (nódulo o lesión no cancerosa) o *maligno* (cáncer de pulmón).
El objetivo es un **prototipo educativo** que muestre cómo un modelo preentrenado puede apoyar el **triaje** (priorizar qué estudios revisa primero un especialista) y cómo un LLM traduce el resultado a lenguaje simple.

| Enfoque | Librería | Modelo | Idea |
|---|---|---|---|
| Línea base | Hugging Face `transformers` | CLIP ViT-B/32 | *Zero-shot*: sin entrenar, solo describiendo las clases en texto |
| Alternativa | `timm` | EfficientNet-B0 | *Transfer learning* sobre ImageNet |
| **Elegido** | `ultralytics` | YOLO-cls nano | *Transfer learning* "en 3 líneas", pensado para producción |

Todos se comparan con **la misma métrica (accuracy balanceada) sobre el mismo conjunto de prueba**.

> ⚠️ Uso exclusivamente educativo. Ningún modelo de este notebook tiene validación clínica y **no entrega diagnósticos**.

El notebook funciona en **Google Colab** (GPU, recomendado) y en local (Mac con GPU Apple `mps` o CPU).
En Colab: `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`. Al terminar, descarga la carpeta `/content/modelo` y `/content/web/public/ejemplos` al repositorio.


## 1. Ficha de datos

| Aspecto | Detalle |
|---|---|
| Dataset | *The IQ-OTH/NCCD lung cancer dataset* — Alyasriy & AL-Huseiny (Iraq-Oncology Teaching Hospital / National Center for Cancer Diseases), 2019–2023 |
| Enlace | https://www.kaggle.com/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset (espejo: https://data.mendeley.com/datasets/bhmdr45bh2) |
| Licencia | CC BY 4.0 |
| Origen | TC de tórax de 110 pacientes recolectadas en dos centros de Irak (otoño 2019); escáner Siemens SOMATOM, 120 kV, cortes de 1 mm |
| Formato | JPG (convertido desde DICOM), escala de grises, cortes axiales |
| Clases | `benign` · `malignant` · `normal` |
| Original | ≈ 1.100–1.200 cortes: 55 casos normales, 40 malignos y 15 benignos |

**Cómo dividimos.** Juntamos todas las imágenes, **eliminamos duplicados exactos** (mismo contenido de archivo) y hacemos una división **estratificada 70 / 15 / 15** (entrenamiento / validación / prueba) con semilla fija. Estratificar mantiene la misma proporción de clases en los tres conjuntos.

**Limitaciones conocidas** (se analizan con datos en la sección 2):
1. **Desbalance fuerte**: la clase *benigno* tiene ~10 veces menos imágenes que *maligno*. Por eso la métrica principal es la **accuracy balanceada** y miramos el recall de cada clase por separado.
2. **Varios cortes por paciente sin identificador fiable**: solo 110 pacientes para más de mil cortes. Como no podemos dividir por paciente, cortes vecinos del mismo paciente quedan en conjuntos distintos → la métrica es **optimista** (el modelo puede "reconocer al paciente" en vez de la lesión).
3. **Dataset pequeño**: con ~170 imágenes de prueba, cada error mueve la métrica casi un 1 %; los intervalos de confianza son anchos.
4. **Población y equipos acotados**: dos centros de un solo país y un solo escáner; no representa a la población chilena ni a otros equipos.


In [ ]:
# ---- 0. Entorno: funciona igual en Colab y en local ----
import sys, os, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    subprocess.run("pip install -q timm ultralytics kagglehub", shell=True)

import hashlib, json, random, shutil, time
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, timm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score)

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
DISP = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
RAIZ = Path("/content") if EN_COLAB else Path.cwd().parent      # raíz del repositorio
DATOS = RAIZ / "datos"                                          # no se sube a GitHub
SALIDA = RAIZ / "modelo"; SALIDA.mkdir(exist_ok=True)                 # lo que usa la app
PESOS_EFF = RAIZ / "runs" / "efficientnet_b0.pt"; PESOS_EFF.parent.mkdir(exist_ok=True)
print("Dispositivo:", DISP, "| torch", torch.__version__, "| timm", timm.__version__)

## 2. Descarga, limpieza de duplicados y división 70/15/15

In [ ]:
import re, kagglehub
origen = Path(kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset"))
CLASES = ["benign", "malignant", "normal"]          # orden alfabético = orden de ImageFolder y de YOLO
# Las carpetas del dataset se llaman "Bengin cases" (sic), "Malignant cases" y "Normal cases";
# las buscamos de forma tolerante por si cambian de nombre o de nivel de anidación.
PATRON = {"benign": r"ben[gi]+n", "malignant": r"malign", "normal": r"normal"}
carpetas = {c: next(d for d in origen.rglob("*") if d.is_dir() and re.search(p, d.name, re.I)) for c, p in PATRON.items()}
print({c: str(d.relative_to(origen)) for c, d in carpetas.items()})

# Hash MD5 del contenido de cada archivo para detectar duplicados exactos
filas = []
for c, carpeta in carpetas.items():
    for f in sorted(carpeta.rglob("*")):
        if f.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            filas.append({"ruta": f, "clase": c, "md5": hashlib.md5(f.read_bytes()).hexdigest()})
df = pd.DataFrame(filas)
print("Imágenes originales:", len(df))

# ¿Hay imágenes idénticas? ¿Alguna aparece con DOS etiquetas distintas?
dup = df[df.duplicated("md5", keep=False)]
conflicto = dup.groupby("md5")["clase"].nunique().gt(1).sum()
print(f"Archivos en grupos duplicados: {len(dup)} | grupos con etiquetas contradictorias: {conflicto}")

# Quitamos las imágenes con etiquetas contradictorias y nos quedamos con una sola copia del resto
md5_malos = dup.groupby("md5").filter(lambda g: g["clase"].nunique() > 1)["md5"].unique()
limpio = df[~df.md5.isin(md5_malos)].drop_duplicates("md5").reset_index(drop=True)
print("Imágenes únicas tras limpiar:", len(limpio))
limpio.clase.value_counts().sort_index()


In [ ]:
# División estratificada: 70 % train, 15 % val, 15 % test
train, resto = train_test_split(limpio, test_size=0.30, stratify=limpio.clase, random_state=SEMILLA)
val, test = train_test_split(resto, test_size=0.50, stratify=resto.clase, random_state=SEMILLA)

# Carpetas datos/<split>/<clase>/ : el formato que entienden timm (ImageFolder) y YOLO
shutil.rmtree(DATOS, ignore_errors=True)
for nombre, parte in [("train", train), ("val", val), ("test", test)]:
    for _, f in parte.iterrows():
        destino = DATOS / nombre / f.clase / f"{f.md5[:12]}.jpg"
        destino.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f.ruta, destino)

tabla = pd.DataFrame({n: p.clase.value_counts() for n, p in [("train", train), ("val", val), ("test", test)]})
tabla["total"] = tabla.sum(1)
tabla["% del total"] = (100 * tabla.total / tabla.total.sum()).round(1)
display(tabla)
tabla[["train", "val", "test"]].plot.bar(figsize=(7, 3), rot=0, color=["#0e7490", "#22d3ee", "#a5f3fc"],
                                          title="Imágenes por clase y conjunto")
plt.tight_layout(); plt.show()

**Desbalance.** Ver tabla: las clases quedan **fuertemente desbalanceadas** (*benigno* es ~10 % del total). Una accuracy simple engañaría: un modelo que nunca dijera "benigno" podría superar el 85 %. Por eso usamos **accuracy balanceada** como métrica principal (azar = 33 %) y revisamos el recall de *benigno* y de *maligno* por separado.


In [ ]:
# Una muestra de cada clase
fig, ejes = plt.subplots(2, 3, figsize=(9, 6))
for j, c in enumerate(CLASES):
    for i, f in enumerate(random.Random(j).sample(sorted((DATOS / "train" / c).iterdir()), 2)):
        im = Image.open(f)
        ejes[i, j].imshow(im, cmap="gray"); ejes[i, j].axis("off")
        ejes[i, j].set_title(f"{c}  {im.size[0]}×{im.size[1]}", fontsize=9)
plt.tight_layout(); plt.show()

## 3. Evaluación común

Los tres enfoques se evalúan sobre el **mismo conjunto de prueba completo** (que ningún modelo vio al entrenar ni al elegir hiperparámetros).
Métrica principal: **accuracy balanceada** = promedio del *recall* de cada clase (azar = 33 %). Cada enfoque solo debe aportar una función que devuelva las probabilidades de las 3 clases para una lista de rutas.

In [ ]:
PRUEBA = [(p, c) for c in CLASES for p in sorted((DATOS / "test" / c).iterdir())]
Y_TEST = np.array([CLASES.index(c) for _, c in PRUEBA])
RESULTADOS = {}

MAL = CLASES.index("malignant")

def evaluar(nombre, probs, tamano_mb):
    """Guarda las métricas de un enfoque a partir de su matriz de probabilidades (n_test × 3)."""
    pred = probs.argmax(1)
    fn = int(((Y_TEST == MAL) & (pred != MAL)).sum())          # maligno clasificado como normal o benigno
    RESULTADOS[nombre] = {"balanced_accuracy": balanced_accuracy_score(Y_TEST, pred),
                          "accuracy": accuracy_score(Y_TEST, pred), "falsos_negativos": fn,
                          "tamano_mb": tamano_mb, "pred": pred, "probs": probs}
    print(f"{nombre}: accuracy balanceada = {RESULTADOS[nombre]['balanced_accuracy']:.1%} "
          f"| accuracy = {RESULTADOS[nombre]['accuracy']:.1%} | falsos negativos (maligno no detectado) = {fn}")


## 4. Línea base · CLIP *zero-shot* (sin entrenar)

CLIP compara la imagen con **frases que nosotros escribimos** y elige la más parecida. No necesita entrenamiento: sirve para medir **cuánto aporta adaptar un modelo a nuestros datos**.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DISP).eval()
clip_prep = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
TEXTOS = {   # una descripción en inglés por clase, en el orden de CLASES
    "benign":    "a chest CT scan showing a small round benign lung nodule with smooth edges",
    "malignant": "a chest CT scan showing a malignant lung cancer tumor with irregular spiculated edges",
    "normal":    "a chest CT scan of healthy normal lungs with no nodules or masses",
}

@torch.no_grad()
def probs_clip(rutas, lote=64):
    salida = []
    for i in range(0, len(rutas), lote):
        imgs = [Image.open(r).convert("RGB") for r in rutas[i:i + lote]]
        x = clip_prep(text=list(TEXTOS.values()), images=imgs, return_tensors="pt", padding=True).to(DISP)
        salida.append(clip(**x).logits_per_image.softmax(1).cpu().numpy())
    return np.concatenate(salida)

evaluar("CLIP zero-shot", probs_clip([p for p, _ in PRUEBA]), tamano_mb=605)

## 5. Enfoque 2 · EfficientNet-B0 con *transfer learning* (`timm`)

1. Cargamos EfficientNet-B0 **preentrenada en ImageNet** y reemplazamos su última capa por una de **3 salidas**.
2. Entrenamos con **aumentación de datos** (recortes, volteos, cambios de brillo) para que el modelo no memorice y tolere variaciones de equipo.
3. Tasa de aprendizaje pequeña (3e-4) con *cosine decay*: ajustamos sin "olvidar" lo que ya sabe.
4. Tras cada época medimos la **accuracy balanceada en validación** y guardamos **el mejor modelo** (no el último).

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

modelo = timm.create_model("efficientnet_b0", pretrained=True, num_classes=len(CLASES)).to(DISP)
cfg = timm.data.resolve_model_data_config(modelo)                 # 224 px + normalización ImageNet
prep_train = timm.data.create_transform(**cfg, is_training=True, hflip=0.5, color_jitter=0.2,
                                        scale=(0.7, 1.0), auto_augment=None, re_prob=0.0)
prep = timm.data.create_transform(**cfg)                          # sin aumentación para val/test/app

ds_train = ImageFolder(DATOS / "train", transform=prep_train)
ds_val = ImageFolder(DATOS / "val", transform=prep)
assert ds_train.classes == CLASES
dl_train = DataLoader(ds_train, batch_size=32, shuffle=True, num_workers=0)
dl_val = DataLoader(ds_val, batch_size=64, num_workers=0)

EPOCAS = 8
opt = torch.optim.AdamW(modelo.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCAS * len(dl_train))
perdida = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

@torch.no_grad()
def predecir_loader(m, dl):
    m.eval(); P, Y = [], []
    for x, y in dl:
        P.append(m(x.to(DISP)).softmax(1).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

historia, mejor = [], 0
for epoca in range(EPOCAS):
    modelo.train(); suma, t0 = 0.0, time.time()
    for x, y in dl_train:
        opt.zero_grad()
        l = perdida(modelo(x.to(DISP)), y.to(DISP))
        l.backward(); opt.step(); sched.step(); suma += l.item()
    p_val, y_val = predecir_loader(modelo, dl_val)
    bacc = balanced_accuracy_score(y_val, p_val.argmax(1))
    historia.append({"epoca": epoca + 1, "perdida_train": suma / len(dl_train), "bacc_val": bacc})
    if bacc > mejor:                                   # guardamos el mejor según VALIDACIÓN
        mejor = bacc; torch.save(modelo.state_dict(), PESOS_EFF)
    print(f"época {epoca + 1} | pérdida {suma / len(dl_train):.3f} | bacc val {bacc:.1%} | {time.time() - t0:.0f}s")

modelo.load_state_dict(torch.load(PESOS_EFF)); modelo.eval()
print(f"Mejor accuracy balanceada en validación: {mejor:.1%}")

In [ ]:
h = pd.DataFrame(historia)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3))
a.plot(h.epoca, h.perdida_train, "o-", color="#0e7490"); a.set_title("Pérdida de entrenamiento"); a.set_xlabel("época")
b.plot(h.epoca, h.bacc_val, "o-", color="#e07b00"); b.set_title("Accuracy balanceada en validación"); b.set_xlabel("época")
plt.tight_layout(); plt.show()

In [ ]:
# Evaluación en PRUEBA
dl_test = DataLoader(ImageFolder(DATOS / "test", transform=prep), batch_size=64, num_workers=0)
p_test, y_chk = predecir_loader(modelo, dl_test)
assert (y_chk == Y_TEST).all()
evaluar("EfficientNet-B0 (timm)", p_test, tamano_mb=os.path.getsize(PESOS_EFF) / 1e6)
cpu = timm.create_model("efficientnet_b0", num_classes=len(CLASES)); cpu.load_state_dict(modelo.state_dict()); cpu.eval()

## 6. Enfoque 3 (elegido) · YOLO-cls nano (`ultralytics`)

YOLO lee directamente la carpeta `datos/` (`train/` y `val/`) y se encarga del entrenamiento y la aumentación.

In [ ]:
from IPython.utils.capture import capture_output
from ultralytics import YOLO
try:
    yolo = YOLO("yolo26n-cls.pt")
except Exception:
    yolo = YOLO("yolo11n-cls.pt")
with capture_output():          # oculta las barras de progreso de YOLO (miles de líneas)
    yolo.train(data=str(DATOS), epochs=10, imgsz=224, batch=32, device=DISP, seed=SEMILLA,
               plots=False, verbose=False, project=str(RAIZ / "runs"), name="yolo", exist_ok=True)
pesos_yolo = Path(yolo.trainer.best)
yolo = YOLO(pesos_yolo)          # recargamos el mejor modelo según validación
print("Mejor modelo YOLO:", pesos_yolo.name)

def probs_yolo(rutas, device=DISP):
    salida = []
    for r in yolo.predict([str(p) for p in rutas], imgsz=224, verbose=False, stream=True, device=device):
        # YOLO ordena sus clases alfabéticamente igual que CLASES
        salida.append(r.probs.data.cpu().numpy())
    return np.stack(salida)

assert list(yolo.names.values()) == CLASES
evaluar("YOLO-cls nano", probs_yolo([p for p, _ in PRUEBA]), tamano_mb=os.path.getsize(pesos_yolo) / 1e6)

## 7. Comparación con la misma métrica

In [ ]:
# Velocidad: los 3 modelos en CPU, de a una imagen (así correrá en el servidor gratuito)
def ms_cpu(predecir, n=20):
    predecir(); t0 = time.time()
    for _ in range(n): predecir()
    return 1000 * (time.time() - t0) / n

ruta1 = PRUEBA[0][0]
with torch.no_grad():
    x1 = prep(Image.open(ruta1).convert("RGB")).unsqueeze(0)
    RESULTADOS["EfficientNet-B0 (timm)"]["ms_por_imagen"] = ms_cpu(lambda: cpu(x1))
    RESULTADOS["YOLO-cls nano"]["ms_por_imagen"] = ms_cpu(lambda: probs_yolo([ruta1], device="cpu"))
    clip.to("cpu"); DISP_CLIP, DISP = DISP, "cpu"
    RESULTADOS["CLIP zero-shot"]["ms_por_imagen"] = ms_cpu(lambda: probs_clip([ruta1]), n=5)
    DISP = DISP_CLIP

comp = pd.DataFrame(RESULTADOS).T[["balanced_accuracy", "accuracy", "falsos_negativos", "tamano_mb", "ms_por_imagen"]]
display(comp.style.format({"balanced_accuracy": "{:.1%}", "accuracy": "{:.1%}", "tamano_mb": "{:.1f}",
                           "ms_por_imagen": "{:.0f}"}))
plt.figure(figsize=(8, 2.6))
plt.barh(comp.index, comp.balanced_accuracy, color=["#9aa5b1", "#0e7490", "#e07b00"])
for i, v in enumerate(comp.balanced_accuracy): plt.text(v + 0.01, i, f"{v:.1%}", va="center")
plt.axvline(1 / 3, ls="--", color="gray"); plt.text(0.34, -0.45, "azar", color="gray")
plt.xlim(0, 1.08); plt.xlabel("Accuracy balanceada en prueba"); plt.tight_layout(); plt.show()

### ¿Por qué elegimos YOLO-cls nano?

- **CLIP sin entrenar queda cerca del azar (≈33 %)**: un modelo general, entrenado con fotos de internet, no sabe distinguir un nódulo benigno de uno maligno en una TC. Adaptar el modelo con datos del dominio es indispensable.
- **YOLO-cls nano y EfficientNet-B0 rinden parecido** (comparar en la tabla de arriba). Los criterios de decisión, en orden, son:
  1. **accuracy balanceada** en prueba (cada clase pesa lo mismo, importante por el desbalance),
  2. **menos falsos negativos** (maligno clasificado como normal/benigno), que es el error más costoso,
  3. **costo de operación**: YOLO es ≈5 veces más liviano (≈3 MB vs ≈16 MB) y mucho más rápido en CPU, lo que importa porque la app corre en una VM sin GPU.
- Con ~170 imágenes de prueba la diferencia de accuracy puede ser de pocas imágenes, así que **el argumento decisivo es el conjunto**: igual o mejor desempeño, menos errores graves y menor costo de operación. Si en tu ejecución EfficientNet gana claramente en falsos negativos, documenta la decisión y considera cambiar el modelo elegido.
- EfficientNet sigue siendo una buena alternativa: Grad-CAM es más directo en `timm`, pero lo implementamos también para YOLO (sección 11).


## 8. Métricas detalladas del modelo elegido en prueba

In [ ]:
p_test = RESULTADOS["YOLO-cls nano"]["probs"]      # desde aquí, todo es sobre el modelo elegido
pred = p_test.argmax(1)
NOMBRES = ["Benigno", "Maligno", "Normal"]
print(classification_report(Y_TEST, pred, target_names=NOMBRES, digits=3))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(confusion_matrix(Y_TEST, pred), display_labels=NOMBRES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_xlabel("Predicción"); ax.set_ylabel("Real"); plt.title("Matriz de confusión · prueba"); plt.tight_layout(); plt.show()

## 9. El error más costoso: un caso maligno clasificado como normal o benigno

En un sistema de apoyo al triaje, **un falso negativo** (decir *normal* o *benigno* cuando el corte muestra cáncer) es el peor error: el estudio podría quedar al final de la fila y retrasar el diagnóstico y tratamiento. Confundir *normal* con *benigno* es mucho menos grave: ambos llegan al especialista con prioridad similar. Un falso positivo (*maligno* en un pulmón sano o con nódulo benigno) genera ansiedad y exámenes extra, pero se corrige al revisar.

Por eso medimos aparte la **sensibilidad para "es maligno"** (maligno vs. no maligno, juntando *normal* y *benigno*).


In [ ]:
MAL = CLASES.index("malignant")
es_maligno, dice_maligno = Y_TEST == MAL, pred == MAL
fn = np.where(es_maligno & ~dice_maligno)[0]; fp = np.where(~es_maligno & dice_maligno)[0]
sens = (es_maligno & dice_maligno).sum() / es_maligno.sum(); esp = (~es_maligno & ~dice_maligno).sum() / (~es_maligno).sum()
print(f"Sensibilidad 'es maligno': {sens:.2%}  | Especificidad: {esp:.2%}")
print(f"Falsos negativos (maligno → normal/benigno): {len(fn)} de {es_maligno.sum()} | Falsos positivos: {len(fp)}")

errores = np.where(pred != Y_TEST)[0]
muestra = list(fn) + [i for i in errores if i not in fn]
fig, ejes = plt.subplots(1, min(6, len(muestra)) or 1, figsize=(14, 3), squeeze=False)
for ax, i in zip(ejes[0], muestra[:6]):
    ax.imshow(Image.open(PRUEBA[i][0]), cmap="gray"); ax.axis("off")
    ax.set_title(f"real {NOMBRES[Y_TEST[i]]}\npred {NOMBRES[pred[i]]} ({p_test[i].max():.0%})", fontsize=9,
                 color="red" if i in fn else "black")
plt.suptitle("Errores en prueba (en rojo: falsos negativos)"); plt.tight_layout(); plt.show()


## 10. Umbral de confianza calibrado en validación

La app avisa **"requiere revisión humana"** cuando la confianza (probabilidad de la clase ganadora) es baja. En vez de inventar un umbral, lo elegimos con el **conjunto de validación**: el **menor umbral** tal que las predicciones por sobre él tengan **≥ 99 % de acierto**. Luego verificamos qué pasa en prueba.

In [ ]:
VALID = [(p, c) for c in CLASES for p in sorted((DATOS / "val" / c).iterdir())]
p_val, y_val = probs_yolo([p for p, _ in VALID]), np.array([CLASES.index(c) for _, c in VALID])
conf_val, ok_val = p_val.max(1), p_val.argmax(1) == y_val
candidatos = np.round(np.arange(0.40, 0.96, 0.01), 2)
acc_sobre = [ok_val[conf_val >= t].mean() for t in candidatos]
cobertura = [(conf_val >= t).mean() for t in candidatos]
UMBRAL = float(next((t for t, a in zip(candidatos, acc_sobre) if a >= 0.99), 0.90))

conf_test, ok_test = p_test.max(1), pred == Y_TEST
print(f"Umbral elegido: {UMBRAL:.2f}")
print(f"Prueba · sobre el umbral: {(conf_test >= UMBRAL).mean():.1%} de las imágenes con {ok_test[conf_test >= UMBRAL].mean():.1%} de acierto")
print(f"Prueba · bajo el umbral (a revisión humana): {(conf_test < UMBRAL).mean():.1%} con {ok_test[conf_test < UMBRAL].mean():.1%} de acierto")
print(f"Falsos negativos que quedan sobre el umbral: {(conf_test[fn] >= UMBRAL).sum()} de {len(fn)}")

plt.figure(figsize=(7, 3))
plt.plot(candidatos, acc_sobre, label="acierto sobre el umbral", color="#0e7490")
plt.plot(candidatos, cobertura, label="% de imágenes sobre el umbral", color="#e07b00")
plt.axvline(UMBRAL, ls="--", color="gray"); plt.xlabel("umbral de confianza"); plt.legend(); plt.title("Calibración en validación")
plt.tight_layout(); plt.show()

## 11. ¿Dónde mira el modelo? Grad-CAM

Grad-CAM pondera los mapas de activación de la última capa convolucional según cuánto influyen en la clase predicha. Es la misma idea de los *mapas de activación* de la Pauta 1, pero para un modelo preentrenado. Para YOLO recorremos su red capa por capa (es una lista de bloques de PyTorch) y nos quedamos con la salida de la última convolución. La app usa exactamente esta función.

In [ ]:
from torchvision import transforms
red = yolo.model.float().eval().to("cpu")
recorte = transforms.Compose([transforms.Resize(224), transforms.CenterCrop(224)])   # igual que YOLO

def gradcam(ruta):
    img = recorte(Image.open(ruta).convert("RGB"))
    x = transforms.ToTensor()(img).unsqueeze(0).requires_grad_()   # pide gradientes desde la entrada
    for capa in red.model[:-1]:            # bloques 0..9: extraen características
        x = capa(x)
    cabeza = red.model[-1]
    act = cabeza.conv(x); act.retain_grad()   # última convolución: 1280 mapas de 7×7
    logits = cabeza.linear(cabeza.pool(act).flatten(1))
    k = logits.argmax(1).item(); logits[0, k].backward()
    cam = torch.relu((act.grad.mean((2, 3), keepdim=True) * act).sum(1))[0]
    cam = torch.nn.functional.interpolate((cam / cam.max())[None, None], size=img.size[::-1], mode="bilinear")[0, 0]
    return img, cam.detach().numpy(), CLASES[k]

fig, ejes = plt.subplots(2, 3, figsize=(9, 6))
for j, c in enumerate(CLASES):
    ruta = next(p for p, cc in PRUEBA if cc == c)
    img, cam, k = gradcam(ruta)
    ejes[0, j].imshow(img); ejes[0, j].set_title(f"real: {c}"); ejes[0, j].axis("off")
    ejes[1, j].imshow(img); ejes[1, j].imshow(cam, cmap="jet", alpha=0.45); ejes[1, j].set_title(f"pred: {k}"); ejes[1, j].axis("off")
plt.tight_layout(); plt.show()

## 12. Exportar el modelo y los archivos que usa la app

In [ ]:
INFO_CLASES = [
    {"id": "benign", "nombre": "Benigno", "color": "#fbbf24",
     "descripcion": "El corte muestra un nódulo o lesión que el modelo asocia con los casos benignos (no cancerosos) del dataset. Igual requiere seguimiento médico."},
    {"id": "malignant", "nombre": "Maligno", "color": "#fb7185",
     "descripcion": "El corte muestra hallazgos que el modelo asocia con cáncer de pulmón. Es la clase donde un error del modelo es más costoso."},
    {"id": "normal", "nombre": "Normal", "color": "#34d399",
     "descripcion": "El modelo no reconoce lesiones de las categorías entrenadas en este corte. No descarta otras condiciones ni cortes vecinos."},
]
(SALIDA / "clases.json").write_text(json.dumps({"clases": INFO_CLASES}, ensure_ascii=False, indent=2), encoding="utf-8")

cm = confusion_matrix(Y_TEST, pred)
metricas = {
    "modelo": "YOLO-cls nano", "accuracy": float(accuracy_score(Y_TEST, pred)),
    "balanced_accuracy": float(balanced_accuracy_score(Y_TEST, pred)), "n_test": int(len(Y_TEST)),
    "n_train": len(train), "n_val": len(val),
    "recall_por_clase": dict(zip(CLASES, map(float, recall_score(Y_TEST, pred, average=None)))),
    "precision_por_clase": dict(zip(CLASES, map(float, precision_score(Y_TEST, pred, average=None)))),
    "matriz_confusion": cm.tolist(), "sensibilidad_maligno": float(sens), "especificidad_maligno": float(esp),
    "falsos_negativos": int(len(fn)), "umbral": UMBRAL,
    "comparacion": [{"modelo": k, "balanced_accuracy": float(v["balanced_accuracy"]), "accuracy": float(v["accuracy"]),
                     "falsos_negativos": int(v["falsos_negativos"]),
                     "tamano_mb": round(float(v["tamano_mb"]), 1), "ms_por_imagen": round(float(v["ms_por_imagen"]), 1)}
                    for k, v in RESULTADOS.items()],
}
(SALIDA / "metricas.json").write_text(json.dumps(metricas, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.copy(pesos_yolo, SALIDA / "yolo_cls.pt")      # pesos del modelo elegido (≈3 MB, cabe en GitHub)

# Imágenes de ejemplo (de PRUEBA) para el repositorio y la app: 1 por clase + 18 para el modo desafío
web = RAIZ / "web" / "public" / "ejemplos"; (web / "desafio").mkdir(parents=True, exist_ok=True)
(RAIZ / "ejemplos").mkdir(exist_ok=True)
rng, desafio = random.Random(SEMILLA), []
for c in CLASES:
    rutas = [p for p, cc in PRUEBA if cc == c]
    for destino in (RAIZ / "ejemplos" / f"{c}.jpg", web / f"{c}.jpg"):
        Image.open(rutas[0]).convert("RGB").save(destino, quality=90)
    desafio += [(r, c) for r in rng.sample(rutas[1:], min(6, len(rutas) - 1))]
rng.shuffle(desafio)
lista = []
for k, (r, c) in enumerate(desafio):    # nombres neutros: el archivo no delata la clase
    img = Image.open(r).convert("RGB"); img.thumbnail((384, 384))
    img.save(web / "desafio" / f"{k:02d}.jpg", quality=85)
    lista.append({"archivo": f"desafio/{k:02d}.jpg", "clase": c})
(web / "desafio.json").write_text(json.dumps(lista, indent=1), encoding="utf-8")
print(json.dumps({k: v for k, v in metricas.items() if k != "comparacion"}, indent=1, ensure_ascii=False)[:800])

## 13. Conclusiones, riesgos y mejoras

**Resultado.** YOLO-cls nano ajustado con *transfer learning* clasifica las 3 clases con una accuracy balanceada alta en prueba (ver sección 7), frente a un CLIP sin entrenar cercano al azar. La lección central del módulo se confirma: **un modelo preentrenado general no basta; adaptarlo con pocos datos del dominio lo cambia todo**.

**Error más costoso.** Los falsos negativos (maligno → normal/benigno) son el riesgo principal; revisar cuántos quedan **sobre el umbral**: si existen, el umbral no los atrapa a todos. Por eso la app nunca dice "descartado": siempre recomienda revisión profesional.

**Umbral.** Calibrado en validación, envía a revisión humana las imágenes de menor confianza, donde el modelo acierta menos: la confianza sí es informativa. Con un conjunto de validación pequeño (~170 imágenes) el umbral es inestable; conviene verificarlo con otra semilla.

**Riesgos / por qué desconfiar de la métrica.**
1. Sin identificador de paciente: cortes vecinos del mismo paciente están en train y test → el modelo puede memorizar al paciente; la métrica es **muy optimista**. Una división por paciente bajaría el desempeño.
2. Desbalance fuerte: *benigno* tiene ~18 imágenes de prueba; su recall es poco confiable.
3. Un solo escáner y dos centros; una imagen fuera de ese mundo (otra patología, una radiografía, una foto cualquiera) igual recibirá una de las 3 etiquetas.

**Mejoras futuras.** Validar con un dataset externo (p. ej. LIDC-IDRI), dividir por paciente, balancear con sobremuestreo o pesos de clase, agregar una clase "fuera de distribución", y evaluar con radiólogos en un estudio real antes de cualquier uso clínico.
